# Complexity-03f — La chaîne des réductions : de 3SUM au triangle lopsided

**Navigation** : [Index](README.md) | [Version Python](Complexity-03f-ReductionsTriangle-Lopsided-Python.ipynb) | [03e — Word RAM](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb)

**Objectifs** : distinguer sens d'une réduction et transfert de dureté ; construire le pont colorié 3SUM–triangle ; mesurer intersections et comparer des formules de coûts. **Prérequis** : complexité, 3SUM et graphes. **Durée estimée** : 60 minutes.

Ce carnet exécute des contrôles naïfs, pas l'algorithme d'Alman et Vassilevska Williams. Les résultats du papier sont cités ; les chronos et corpus sont mesurés. `System.Random` diffère du générateur Python.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
CultureInfo.CurrentCulture=CultureInfo.InvariantCulture;
const int SEED=20261111; var rng=new Random(SEED);
Console.WriteLine($"{System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription} ; graine {SEED}");

.NET 10.0.14 ; graine 20261111


## Trois hypothèses et une toile

Les hypothèses 3SUM et APSP postulaient respectivement des barrières $n^{2-o(1)}$ et $n^{3-o(1)}$ ; SETH concerne les algorithmes SAT exponentiels. Une réduction de $A$ vers $B$ dit qu'une amélioration adéquate pour $B$ entraîne une amélioration pour $A$. Elle transmet donc une dureté supposée de $A$ vers $B$ et permet aussi de remonter une amélioration de $B$ vers $A$.

Le papier cité relie 3SUM, APSP et k-Clique au triangle exact, puis aux triangles creux et lopsided ; SETH et Orthogonal Vectors ne sont pas réfutées par cette chaîne.

In [2]:
(bool ok,int[] triple,int ops) TroisSommes(int[] xs) {
 var a=xs.OrderBy(x=>x).ToArray();int ops=0;
 for(int i=0;i<a.Length-2;i++) {int lo=i+1,hi=a.Length-1;long cible=-(long)a[i];while(lo<hi){ops++;long s=(long)a[lo]+a[hi];if(s==cible)return(true,new[]{a[i],a[lo],a[hi]},ops);if(s<cible)lo++;else hi--;}}
 return(false,Array.Empty<int>(),ops);
}
bool Oracle(int[] a) {for(int i=0;i<a.Length;i++)for(int j=i+1;j<a.Length;j++)for(int k=j+1;k<a.Length;k++)if((long)a[i]+a[j]+a[k]==0)return true;return false;}
int[] Instance(int n,bool oui,int borne,Random r) {var pool=Enumerable.Range(-borne,2*borne+1).OrderBy(_=>r.Next()).Take(n).ToArray(); if(oui) {int a=r.Next(1,borne/2),b; do {b=r.Next(1,borne/2);}while(a==b);pool[0]=a;pool[1]=b;pool[2]=-a-b;} return pool;}
int nDemo=60;var oui=Instance(nDemo,true,500,rng);var non=Enumerable.Range(0,nDemo).Select(k=>3*k+1).ToArray(); var ro=TroisSommes(oui);var rn=TroisSommes(non);
if(!ro.ok || !Oracle(oui) || rn.ok || Oracle(non) || rn.ops!=(nDemo-1)*(nDemo-2)/2) throw new Exception("Contrôle 3SUM incorrect");
Console.WriteLine($"Avec solution : {ro.ok}, triple [{string.Join(",",ro.triple)}], {ro.ops} comparaisons"); Console.WriteLine($"Sans solution : {rn.ok}, {rn.ops} comparaisons ; formule exacte {(nDemo-1)*(nDemo-2)/2}");

Avec solution : True, triple [-498,157,341], 104 comparaisons


Sans solution : False, 1711 comparaisons ; formule exacte 1711


Pour l'entrée positive sans solution, chaque ancre épuise les pointeurs : $(n-1)(n-2)/2$ comparaisons, soit asymptotiquement $n^2/2$. Cela décrit cet algorithme, pas une preuve de la conjecture ni une exécution de la borne sous-quadratique citée.

## De 3SUM au triangle exact

Pour trois ensembles $A,B,C$, les poids des arêtes AB, BC, CA valent respectivement $a,b,c$ ; chaque triangle a donc pour poids $a+b+c$. Colorier indépendamment les éléments d'un ensemble en trois couleurs capture un triple d'indices fixé avec probabilité $6/27=2/9$ ; $r$ coloriages le manquent avec probabilité $(7/9)^r$. Le pont est pédagogique et aléatoire ; le papier cite d'autres réductions dérandomisées.

In [3]:
record Triparti(int[] A,int[] B,int[] C);
Triparti Colorier(int[] xs,Random r) {var groupes=new[]{new List<int>(),new List<int>(),new List<int>()};foreach(int x in xs)groupes[r.Next(3)].Add(x);return new(groupes[0].ToArray(),groupes[1].ToArray(),groupes[2].ToArray());}
bool Triangle(Triparti g) {foreach(int a in g.A)foreach(int b in g.B)foreach(int c in g.C)if((long)a+b+c==0)return true;return false;}
int accords=0,fantomes=0,capturesOui=0;
for(int i=0;i<40;i++){int taille=rng.Next(12,22);var x=i%2==0?Instance(taille,true,120,rng):Enumerable.Range(0,taille).Select(k=>3*k+1).ToArray();bool attendu=Oracle(x);if(attendu!=(i%2==0))throw new Exception("Corpus incorrect");bool vu=false;for(int j=0;j<30;j++){if(Triangle(Colorier(x,rng))){vu=true;break;}}if(vu==attendu)accords++;if(vu&&!attendu)fantomes++;if(vu&&attendu)capturesOui++;}
Console.WriteLine($"Corpus : {accords}/40 accords, {fantomes} fantômes ; {capturesOui}/20 positifs capturés en 30 coloriages.");

Corpus : 40/40 accords, 0 fantômes ; 20/20 positifs capturés en 30 coloriages.


Le corpus donne 40 accords sur 40, dont 20 captures positives, et aucun fantôme parmi les 20 négatifs. C’est un contrôle fini, non une preuve de capture garantie par 30 coloriages.

### Exercice 1 — côté négatif du pont

Écrire un prédicat pour détecter un triangle de somme nulle dans les trois parties. Les fixtures comprennent des cas positifs et négatifs ; une fois complété, le contrôle génère 10 instances sans triple et 30 coloriages chacune.

In [4]:
bool? TriangleEtudiant(Triparti g) { // TODO : chercher un élément dans chaque partie
 return null;
}
var essai=TriangleEtudiant(new Triparti(new[]{1,7},new[]{2},new[]{-3}));
if(essai is null) Console.WriteLine("Exercice 1 à compléter : triangle exact et contrôle négatif.");
else {var faux=TriangleEtudiant(new Triparti(new[]{1,4},new[]{2},new[]{3}));var vide=TriangleEtudiant(new Triparti(Array.Empty<int>(),new[]{2},new[]{-2}));if(essai!=true||faux!=false||vide!=false)throw new Exception("Prédicat incorrect");int fantomesEx=0;for(int i=0;i<10;i++){var xs=Enumerable.Range(0,12+i).Select(k=>3*k+1).ToArray();if(Oracle(xs))throw new Exception("Instance négative incorrecte");for(int j=0;j<30;j++){var g=Colorier(xs,rng);var t=TriangleEtudiant(g);if(t is null || t.Value!=Triangle(g))throw new Exception("Triangle incorrect sur le corpus");if(t.Value)fantomesEx++;}}Console.WriteLine($"10 instances sans solution × 30 coloriages : {fantomesEx} fantômes.");}

Exercice 1 à compléter : triangle exact et contrôle négatif.


In [5]:
var rng2=new Random(SEED+1);int captures=0;const int essais=300, coloriages=5;
for(int i=0;i<essais;i++){var x=Instance(15,true,120,rng2);bool vu=false;for(int j=0;j<coloriages;j++)if(Triangle(Colorier(x,rng2))){vu=true;break;}if(vu)captures++;}
double theorie=1-Math.Pow(7.0/9,coloriages);Console.WriteLine($"Captures : {captures}/{essais} = {(double)captures/essais:F3} ; triple fixé : 1-(7/9)^5 = {theorie:F3}.");

Captures : 273/300 = 0.910 ; triple fixé : 1-(7/9)^5 = 0.715.


La probabilité théorique concerne un triple fixé. Une instance peut contenir plusieurs triples et offre alors d'autres chances de capture ; la fréquence empirique est une estimation, pas une borne prouvée par 300 essais. Ici 273 captures sur 300 donnent 0,910, tandis que la probabilité calculée pour un triple fixé vaut 0,715 ; la présence possible de plusieurs triples explique pourquoi les événements ne sont pas identiques.

## All-Edges Sparse Triangle et intersections

Pour chaque arête, chercher un voisin commun revient à détecter un triangle incident. Les bornes citées sont $O(m^{3/2})$ par énumération et $O(m^{2\omega/(\omega+1)})$ par partage ; à $\omega=2$, cette dernière donne $m^{4/3}$. Si le degré moyen est de l'ordre de $\sqrt n$, alors $m=\Theta(n^{3/2})$, d'où les ordres $n^{9/4}$ et $n^2$ respectivement.

Le graphe généré ci-dessous ne contient que des arêtes entre parties externes et milieu : il est biparti et n'a aucun triangle. Les intersections des deux parties externes calculent des entrées d'un produit, pas des triangles déjà présents.

In [6]:
Dictionary<int,HashSet<int>> Graphe(int N,int D,double densite,Random r) {var adj=Enumerable.Range(0,2*N+D).ToDictionary(i=>i,_=>new HashSet<int>());for(int p=0;p<2*N;p++)for(int m=2*N;m<2*N+D;m++)if(r.NextDouble()<densite){adj[p].Add(m);adj[m].Add(p);}return adj;}
Dictionary<(int,int),bool> AretesTriangle(Dictionary<int,HashSet<int>> g) {var res=new Dictionary<(int,int),bool>();foreach(var (u,vs) in g)foreach(int v in vs)if(u<v)res[(u,v)]=vs.Overlaps(g[v]);return res;}
const int N=40,D=6; const double dens=0.55;var adj=Graphe(N,D,dens,rng);var sw=Stopwatch.StartNew();var inter=new int[N,N];int positifs=0;for(int a=0;a<N;a++)for(int b=0;b<N;b++){inter[a,b]=adj[a].Intersect(adj[N+b]).Count();if(inter[a,b]>0)positifs++;}sw.Stop();
var ar=AretesTriangle(adj);if(ar.Values.Any(x=>x))throw new Exception("Le graphe biparti ne peut contenir de triangle");Console.WriteLine($"N={N}, D={D}, densité {dens:F2} : {N*N} paires, {positifs} avec milieu commun ; {ar.Count} arêtes, aucun triangle ; intersections {sw.Elapsed.TotalMilliseconds:F2} ms.");

N=40, D=6, densité 0.55 : 1600 paires, 1420 avec milieu commun ; 268 arêtes, aucun triangle ; intersections 6.12 ms.


Sur cette génération, 1 420 des 1 600 paires externes ont au moins un voisin commun ; les 268 arêtes du graphe biparti ne sont dans aucun triangle. Ajouter une arête entre parties externes pourrait fermer des triangles.

### Exercice 2 — compter les triangles par arête

Retourner un dictionnaire pour toutes les arêtes non orientées, avec leur nombre de voisins communs. Le graphe lopsided est un contrôle dégénéré ; un triangle et un sommet isolé forcent aussi des comptes non nuls.

In [7]:
Dictionary<(int,int),int> CompterTriangles(Dictionary<int,HashSet<int>> g) { // TODO : une entrée par arête canonique u<v
 return null;
}
var fixture=new Dictionary<int,HashSet<int>>{{0,new(){1,2}},{1,new(){0,2}},{2,new(){0,1}},{3,new()}};
var t=CompterTriangles(fixture);
if(t is null)Console.WriteLine("Exercice 2 à compléter : triangles par arête.");
else {if(t.Count!=3 || !new[]{(0,1),(0,2),(1,2)}.All(e=>t.TryGetValue(e,out int v)&&v==1))throw new Exception("Triangle fixture incorrect");var z=CompterTriangles(adj);if(z is null||z.Count!=ar.Count||ar.Keys.Any(e=>!z.TryGetValue(e,out int v)||v!=0))throw new Exception("Graphe biparti incorrect");Console.WriteLine($"Triangle : 3 arêtes de compte 1 ; lopsided : {z.Count} arêtes de compte 0.");}

Exercice 2 à compléter : triangles par arête.


## Produit fin et entrées voulues

Pour $X\in\mathbb Z^{N\times D}$ et $Y\in\mathbb Z^{D\times N}$, le résultat cité demande $D\le N^{1/18}$ et $|W|\le N^2/\sqrt D$, et donne $O(N^2/D^{0,063})$ opérations pour les entrées voulues. Calcul naïf voulu : $|W|D$ ; produit complet : $N^2D$. Au plafond, le rapport des expressions vaut $D^{0,563}$, sans prédire de rapport de temps à cause des constantes cachées.

In [8]:
Console.WriteLine("N      D    plafond W     naïf W.D    complet N².D    N²/D^0.063");
foreach(int taille in new[]{64,128,256,512,1024}) {int d=Math.Max(2,(int)Math.Round(Math.Pow(taille,1.0/18)));int w=(int)(taille*taille/Math.Sqrt(d)); Console.WriteLine($"{taille,4} {d,6} {w,12} {w*d,13} {taille*taille*d,15} {taille*taille/Math.Pow(d,0.063),14:F0}");}
Console.WriteLine("Formules illustratives : D=2 hors domaine pour toutes ces tailles.");

N      D    plafond W     naïf W.D    complet N².D    N²/D^0.063


  64      2         2896          5792            8192           3921


 128      2        11585         23170           32768          15684


 256      2        46340         92680          131072          62736


 512      2       185363        370726          524288         250943


1024      2       741455       1482910         2097152        1003772


Formules illustratives : D=2 hors domaine pour toutes ces tailles.


Ici $D=2$ et $N<2^{18}$ : les lignes sont hors du domaine d'application cité. À $D=2$ le rapport des expressions est environ 1,48 ; il dépasse 2 pour $D>2^{1/0,563}\approx3,43$. Aucune ligne ne mesure le moteur du papier.

### Exercice 3 — régime du wanted set

Calculer pour $\varepsilon\in\{1/36,1/18\}$ et $N\in\{64,256,1024\}$ : $D=N^\varepsilon$, $|W|_{max}/N^2=1/\sqrt D$ et le rapport $D^{0,563}$. Discuter l'arrondi de $D$ et revérifier le domaine.

In [9]:
(double fraction,double rapport)? Regime(double n,double epsilon) { // TODO : calculer les deux quantités
 return null;
}
var ex=Regime(256,1.0/18);
if(ex is null)Console.WriteLine("Exercice 3 à compléter : régime du wanted set.");
else {foreach(var (n,e) in new[]{(64.0,1.0/36),(256.0,1.0/18),(1024.0,1.0/36),(1024.0,1.0/18)}){var v=Regime(n,e);double d=Math.Pow(n,e),fraction=1/Math.Sqrt(d),rapport=Math.Pow(d,0.563);if(v is null||!double.IsFinite(v.Value.fraction)||!double.IsFinite(v.Value.rapport)||Math.Abs(v.Value.fraction-fraction)>1e-9||Math.Abs(v.Value.rapport-rapport)>1e-9)throw new Exception("Régime incorrect");}foreach(double e in new[]{1.0/36,1.0/18})foreach(int n in new[]{64,256,1024}){var v=Regime(n,e);if(v is null||!double.IsFinite(v.Value.fraction)||!double.IsFinite(v.Value.rapport))throw new Exception("Régime manquant");double d=Math.Pow(n,e);if(Math.Abs(v.Value.fraction-1/Math.Sqrt(d))>1e-9||Math.Abs(v.Value.rapport-Math.Pow(d,0.563))>1e-9)throw new Exception("Régime incorrect");Console.WriteLine($"epsilon={e:F5}, N={n}, fraction={v.Value.fraction:F4}, rapport={v.Value.rapport:F4}");}}

Exercice 3 à compléter : régime du wanted set.


## Ce qui survit, ce qui tombe

Les statuts ci-dessous suivent le papier cité (octobre 2026), sans exécution de ses algorithmes. Les bornes en modèles restreints peuvent rester valables quand le nouvel algorithme sort de ces modèles.

In [10]:
var verdicts=new[]{("3SUM","n^1.9992 déterministe / n^1.998 LV","réfutée"),("APSP","n^2.9995 / n^2.998 LV","réfutée"),("Triangle exact","n^2.9983","dureté conditionnelle réfutée"),("produit min-plus","n^2.9995","réfutée"),("k-Clique poids nul","n^(k-epsilon)","réfutée"),("OMv indice fin tau<1/18","pré-calcul n^(2-0.063 tau)","réfutée"),("SETH","—","intacte"),("Orthogonal Vectors","—","intacte"),("3-LDT 3SUM","hors modèle","intacte"),("SLP min-plus","hors modèle","intacte")};foreach(var (objet,borne,statut) in verdicts)Console.WriteLine($"{objet,-26} {borne,-35} {statut}");

3SUM                       n^1.9992 déterministe / n^1.998 LV  réfutée


APSP                       n^2.9995 / n^2.998 LV               réfutée


Triangle exact             n^2.9983                            dureté conditionnelle réfutée


produit min-plus           n^2.9995                            réfutée


k-Clique poids nul         n^(k-epsilon)                       réfutée


OMv indice fin tau<1/18    pré-calcul n^(2-0.063 tau)          réfutée


SETH                       —                                   intacte


Orthogonal Vectors         —                                   intacte


3-LDT 3SUM                 hors modèle                         intacte


SLP min-plus               hors modèle                         intacte


## À retenir

Le double balayage contrôle le point de départ ; les coloriages illustrent le pont vers le triangle exact et les intersections illustrent le produit. Les formules du produit fin et les dix statuts restent cités, non exécutés comme un nouvel algorithme. Les trois exercices explorent le côté négatif, les triangles par arête et le régime wanted.

## Vérifiez votre compréhension

1. Dans quel sens une réduction transmet-elle la dureté ? 2. Pourquoi la probabilité vaut-elle 2/9 ? 3. Pourquoi le tableau de formules ne mesure-t-il pas un gain ?

<details><summary>Réponses</summary>

1. Une réduction de A vers B transmet une dureté de A à B ; une amélioration de B remonte vers A. 2. Six des 27 coloriages sont bijectifs pour un triple fixé. 3. Les dimensions illustratives sont hors domaine et les constantes cachées sont inconnues.
</details>

## Pour aller plus loin

- [Article Alman et Vassilevska Williams, arXiv 2610.06783](https://arxiv.org/abs/2610.06783).
- [03e — Word RAM](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb), [03d — ETH](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb), [07c — bilan](Complexity-07c-LeBilanMathocalypse-Python.ipynb).

**Navigation** : [Index](README.md) | [Version Python](Complexity-03f-ReductionsTriangle-Lopsided-Python.ipynb)